# Internal Analysis on Colab or VSCode+Colab

This notebook runs the first mechanism-analysis pipeline after training:

- hidden states extraction
- layer-wise similarity
- activation patching
- figure export for reports and papers


## Requirements

Before running:

1. Make sure the project directory is available in the runtime.
2. Make sure the trained adapter checkpoint exists.
3. If you use `meta-llama/Llama-3.1-8B-Instruct`, provide an `HF_TOKEN`.
4. If you want patching on `English correct / Fake wrong` pairs, make sure `test_predictions.jsonl` already exists.


In [ ]:
#@title Analysis Parameters
REPO_URL = ""
REPO_BRANCH = "main"
PROJECT_DIR = "/content/MLLMs_Final_Project"
EXISTING_PROJECT_DIR = ""
PROJECT_ARCHIVE_PATH = ""
ARCHIVE_EXTRACT_ROOT = "/content"

BASE_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
RUN_NAME = "main_qlora"
CHECKPOINT_ROOT = ""
PREDICTIONS_ROOT = ""
HF_TOKEN = ""

HIDDEN_STATE_BATCH_SIZE = 2
PATCHING_MAX_PAIRS = 30
PATCHING_PER_RELATION_MAX = 10
PATCHING_TOP_K = 5
USE_DRIVE = False
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/english_fake_transfer"


In [ ]:
!pip install -q -U transformers peft datasets accelerate bitsandbytes pyyaml tqdm huggingface_hub sentencepiece matplotlib


In [ ]:
from pathlib import Path
import os
import subprocess
import zipfile


def locate_project_dir() -> Path:
    candidates = []
    if EXISTING_PROJECT_DIR.strip():
        candidates.append(Path(EXISTING_PROJECT_DIR.strip()))
    candidates.append(Path(PROJECT_DIR))
    candidates.append(Path.cwd())

    for candidate in candidates:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate

    archive_path = Path(PROJECT_ARCHIVE_PATH.strip()) if PROJECT_ARCHIVE_PATH.strip() else None
    if archive_path and archive_path.exists():
        extract_root = Path(ARCHIVE_EXTRACT_ROOT)
        extract_root.mkdir(parents=True, exist_ok=True)
        if not zipfile.is_zipfile(archive_path):
            raise ValueError(f'PROJECT_ARCHIVE_PATH is not a valid zip file: {archive_path}')
        with zipfile.ZipFile(archive_path, 'r') as zf:
            zf.extractall(extract_root)
        for candidate in extract_root.rglob('src'):
            project_candidate = candidate.parent
            if (project_candidate / 'configs').exists():
                return project_candidate

    if REPO_URL.strip():
        clone_target = Path(PROJECT_DIR)
        if clone_target.exists() and (clone_target / '.git').exists():
            subprocess.run(['git', '-C', str(clone_target), 'fetch', 'origin'], check=True)
            subprocess.run(['git', '-C', str(clone_target), 'checkout', REPO_BRANCH], check=True)
            subprocess.run(['git', '-C', str(clone_target), 'pull', 'origin', REPO_BRANCH], check=True)
        else:
            subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(clone_target)], check=True)
        return clone_target

    raise ValueError(
        'Project directory was not found in the runtime. '
        'Set EXISTING_PROJECT_DIR, or set PROJECT_ARCHIVE_PATH, or provide REPO_URL.'
    )


PROJECT_DIR_RESOLVED = locate_project_dir().resolve()
os.chdir(PROJECT_DIR_RESOLVED)
print('project_dir =', PROJECT_DIR_RESOLVED)
print('contents =', sorted([p.name for p in PROJECT_DIR_RESOLVED.iterdir()])[:20])


In [ ]:
import os
from huggingface_hub import login

hf_token = HF_TOKEN.strip() or os.environ.get("HF_TOKEN", "").strip()
try:
    from google.colab import userdata  # type: ignore
    hf_token = hf_token or (userdata.get("HF_TOKEN") or "").strip()
except Exception:
    pass

if hf_token:
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face login complete.")
else:
    print("HF_TOKEN not provided. Continuing without login if the model is public or already cached.")


In [ ]:
from pathlib import Path
import yaml

project_dir = Path(PROJECT_DIR_RESOLVED)
generated_dir = project_dir / 'configs' / 'generated'
generated_dir.mkdir(parents=True, exist_ok=True)

run_root = Path('/content') / 'runs' / RUN_NAME
adapter_path = CHECKPOINT_ROOT.strip() or str(run_root / 'checkpoints')
predictions_root = Path(PREDICTIONS_ROOT) if PREDICTIONS_ROOT.strip() else run_root / 'predictions' / 'default'
predictions_file = str(predictions_root / 'test_predictions.jsonl')

hidden_cfg = {
    'experiment_name': 'english_to_fake_transfer_internal_hidden_states',
    'seed': 13,
    'model': {
        'base_model_name_or_path': BASE_MODEL_NAME,
        'adapter_path': adapter_path,
        'trust_remote_code': False,
        'torch_dtype': 'bfloat16',
        'padding_side': 'left',
    },
    'quantization': {
        'enabled': True,
        'load_in_4bit': True,
        'bnb_4bit_quant_type': 'nf4',
        'bnb_4bit_use_double_quant': True,
        'bnb_4bit_compute_dtype': 'bfloat16',
    },
    'data': {
        'processed_dir': 'data/processed/english_to_fake_transfer',
        'splits': ['dev', 'test'],
    },
    'analysis': {
        'batch_size': HIDDEN_STATE_BATCH_SIZE,
    },
    'output': {
        'hidden_states_dir': 'outputs/analysis/llama31_8b_main_qlora/hidden_states',
        'run_name': RUN_NAME,
    },
    'sft': {'system_prompt': ''},
}

similarity_cfg = {
    'experiment_name': 'english_to_fake_transfer_internal_similarity',
    'seed': 13,
    'input': {
        'hidden_states_dir': 'outputs/analysis/llama31_8b_main_qlora/hidden_states',
        'run_name': RUN_NAME,
        'splits': ['dev', 'test'],
    },
    'analysis': {
        'positions': ['final_token', 'subject_pool', 'relation_pool'],
    },
    'output': {
        'results_dir': 'outputs/analysis/llama31_8b_main_qlora',
        'run_name': RUN_NAME,
    },
}

patching_cfg = {
    'experiment_name': 'english_to_fake_transfer_internal_patching',
    'seed': 13,
    'model': {
        'base_model_name_or_path': BASE_MODEL_NAME,
        'adapter_path': adapter_path,
        'trust_remote_code': False,
        'torch_dtype': 'bfloat16',
        'padding_side': 'left',
    },
    'quantization': {
        'enabled': True,
        'load_in_4bit': True,
        'bnb_4bit_quant_type': 'nf4',
        'bnb_4bit_use_double_quant': True,
        'bnb_4bit_compute_dtype': 'bfloat16',
    },
    'data': {
        'processed_dir': 'data/processed/english_to_fake_transfer',
        'split': 'test',
    },
    'input': {
        'predictions_file': predictions_file,
    },
    'selection': {
        'mode': 'english_correct_fake_wrong',
        'stratify_by_relation': True,
        'target_relations': ['lives_in', 'discovered', 'symbol_is'],
        'min_pairs_per_relation': PATCHING_PER_RELATION_MAX,
        'per_relation_max_pairs': PATCHING_PER_RELATION_MAX,
        'max_pairs_total': PATCHING_MAX_PAIRS,
    },
    'analysis': {
        'positions': ['final_token', 'subject_last_token', 'relation_last_token'],
        'top_k': PATCHING_TOP_K,
    },
    'output': {
        'results_dir': 'outputs/analysis/llama31_8b_main_qlora/patching',
        'run_name': RUN_NAME,
    },
    'sft': {'system_prompt': ''},
}

main_metrics_file = Path('/content') / 'runs' / RUN_NAME / 'metrics' / 'default' / 'metrics.json'
if not main_metrics_file.exists():
    main_metrics_file = project_dir / 'outputs' / 'metrics' / 'llama31_8b_main_qlora' / 'default' / 'metrics.json'

fake_control_metrics_file = Path('/content') / 'runs' / f'{RUN_NAME}_fake_control' / 'metrics' / 'default' / 'metrics.json'
if not fake_control_metrics_file.exists():
    fake_control_metrics_file = project_dir / 'outputs' / 'metrics' / 'llama31_8b_fake_control_qlora' / 'default' / 'metrics.json'

plot_cfg = {
    'experiment_name': 'english_to_fake_transfer_main_plots',
    'input': {
        'split': 'test',
        'run_name': RUN_NAME,
        'main_metrics_file': str(main_metrics_file),
        'fake_control_metrics_file': str(fake_control_metrics_file),
        'similarity_dir': 'outputs/analysis/llama31_8b_main_qlora',
        'patching_dir': 'outputs/analysis/llama31_8b_main_qlora/patching',
    },
    'output': {
        'output_dir': 'outputs/figures/llama31_8b_main_qlora/' + RUN_NAME,
    },
}

configs = {
    generated_dir / 'analysis_hidden_states_colab.yaml': hidden_cfg,
    generated_dir / 'analysis_similarity_colab.yaml': similarity_cfg,
    generated_dir / 'analysis_patching_colab.yaml': patching_cfg,
    generated_dir / 'plot_main_colab.yaml': plot_cfg,
}

for path, payload in configs.items():
    path.write_text(yaml.safe_dump(payload, sort_keys=False, allow_unicode=True), encoding='utf-8')
    print('wrote', path)


In [ ]:
!python src/extract_hidden_states.py --config configs/generated/analysis_hidden_states_colab.yaml


In [ ]:
!python src/analyze_hidden_states.py --config configs/generated/analysis_similarity_colab.yaml


In [ ]:
!python src/activation_patching.py --config configs/generated/analysis_patching_colab.yaml


In [ ]:
!python src/plot_main_results.py --config configs/generated/plot_main_colab.yaml


In [ ]:
from pathlib import Path
fig_root = Path(PROJECT_DIR_RESOLVED) / "outputs" / "figures" / "llama31_8b_main_qlora" / RUN_NAME
print("figure_root =", fig_root)
for path in sorted(fig_root.rglob("*.png")):
    print(path)


In [ ]:
from pathlib import Path
from IPython.display import Image, display

fig_root = Path(PROJECT_DIR_RESOLVED) / "outputs" / "figures" / "llama31_8b_main_qlora" / RUN_NAME
preview_paths = sorted(fig_root.rglob("*.png"))[:6]
print("preview_count =", len(preview_paths))
for path in preview_paths:
    print(path.name)
    display(Image(filename=str(path)))


In [ ]:
from pathlib import Path
import shutil

if USE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception:
        print("Drive mount skipped or unavailable.")

    src_root = Path(PROJECT_DIR_RESOLVED) / "outputs" / "analysis"
    dst_root = Path(DRIVE_OUTPUT_ROOT) / "analysis"
    dst_root.parent.mkdir(parents=True, exist_ok=True)
    if dst_root.exists():
        shutil.rmtree(dst_root)
    shutil.copytree(src_root, dst_root)
    print("copied analysis outputs to", dst_root)
else:
    print("USE_DRIVE is False; skipping copy.")
